# Practical-12: Comparing k-NN, Decision Tree, Naive Bayes and Perceptron

**Aim:** To implement simple AI learning models (k-NN, Decision Tree, Naive Bayes, Perceptron) and compare their results on a toy dataset.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import Perceptron
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

In [ ]:
# Toy dataset: Iris (150 samples, 4 features, 3 classes)
iris = load_iris()
X, y = iris.data, iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Scaling helps k-NN and Perceptron
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print("Training samples:", X_train.shape[0], "| Test samples:", X_test.shape[0])
print("Classes:", list(iris.target_names))

In [ ]:
models = {
    "k-NN (k=5)":     (KNeighborsClassifier(n_neighbors=5), True),
    "Decision Tree":  (DecisionTreeClassifier(random_state=42), False),
    "Naive Bayes":    (GaussianNB(), False),
    "Perceptron":     (Perceptron(max_iter=1000, random_state=42), True),
}

results = []
predictions = {}
for name, (model, needs_scaling) in models.items():
    Xtr = X_train_s if needs_scaling else X_train
    Xte = X_test_s if needs_scaling else X_test
    model.fit(Xtr, y_train)
    y_pred = model.predict(Xte)
    predictions[name] = y_pred
    results.append({
        "Model": name,
        "Accuracy":  accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average="macro"),
        "Recall":    recall_score(y_test, y_pred, average="macro"),
        "F1-score":  f1_score(y_test, y_pred, average="macro"),
    })

results_df = pd.DataFrame(results).set_index("Model").round(4)
results_df

In [ ]:
# Confusion matrices
for name, y_pred in predictions.items():
    print(f"\n{name} - Confusion Matrix")
    print(confusion_matrix(y_test, y_pred))

In [ ]:
# Comparison chart
results_df["Accuracy"].plot(kind="bar", color=["#4c72b0", "#55a868", "#c44e52", "#8172b2"], figsize=(7, 4))
plt.title("Model Comparison on Iris Dataset (Test Accuracy)")
plt.ylabel("Accuracy")
plt.ylim(0, 1.05)
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

best = results_df["Accuracy"].idxmax()
print("Best model by accuracy:", best, "->", results_df.loc[best, "Accuracy"])